Cell 1 — Setup

In [ ]:
import os
import glob
import pandas as pd
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Base paths
BASE_PATH = "/content/drive/MyDrive/Transcripts_CSS"

OUTPUT_DIR = os.path.join(BASE_PATH, "outputs")
LDA_DIR = os.path.join(OUTPUT_DIR, "LDA_results")
LDA_FINAL_DF_DIR = os.path.join(LDA_DIR, "final_df")

# Input files
MAIN_DF_PATH = os.path.join(OUTPUT_DIR, "df_new.csv")
STANZA_DF_PATH = os.path.join(OUTPUT_DIR, "df-stanza-features.csv")
GENDER_DF_PATH = os.path.join(OUTPUT_DIR, "gender_results.csv")

# Final output
ALL_FEATURES_PATH = os.path.join(OUTPUT_DIR, "df-all-features.csv")

# Community name mapping
COMMUNITY_SHORT_NAME = {
    "transcripts_business": "business",
    "transcripts_religion": "religion",
    "transcripts_comedy": "comedy",
    "transcripts_lifestyle": "lifestyle",
    "transcripts_tech": "tech",
    "politics_transcripts": "politics",
    "gaming_transcripts": "gaming",
    "motivational_transcripts": "motivational",
}

pd.set_option("display.max_columns", None)

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Setup complete.")
print(f"OUTPUT_DIR: {OUTPUT_DIR}")
print(f"Stanza features: {STANZA_DF_PATH}")
print(f"Main df: {MAIN_DF_PATH}")
print(f"Gender results: {GENDER_DF_PATH}")
print(f"LDA directory: {LDA_FINAL_DF_DIR}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Setup complete.
OUTPUT_DIR: /content/drive/MyDrive/Transcripts_CSS/outputs
Stanza features: /content/drive/MyDrive/Transcripts_CSS/outputs/df-stanza-features.csv
Main df: /content/drive/MyDrive/Transcripts_CSS/outputs/df_new.csv
Gender results: /content/drive/MyDrive/Transcripts_CSS/outputs/gender_results.csv
LDA directory: /content/drive/MyDrive/Transcripts_CSS/outputs/LDA_results/final_df


Cell 2 — Load main df.csv, stanza features and gender results

In [ ]:
# Load main df
df = pd.read_csv(MAIN_DF_PATH)
print(f"Main df: {df.shape}")
print(df["community"].value_counts())

# Load Stanza features generated by get_stanza_df.py
stanza_features_df = pd.read_csv(STANZA_DF_PATH)
print(f"\nStanza features df: {stanza_features_df.shape}")

# Load gender results
gender_df = pd.read_csv(GENDER_DF_PATH)
gender_df = gender_df.rename(columns={"file": "video_id"})
print(f"Gender results: {gender_df.shape}")


Main df: (9206, 11)
community
transcripts_tech            1434
transcripts_politics        1357
transcripts_gaming          1157
transcripts_lifestyle       1113
transcripts_comedy          1063
transcripts_business        1043
transcripts_religion        1033
transcripts_motivational    1006
Name: count, dtype: int64

Stanza features df: (9206, 75)
Gender results: (9916, 4)


Cell 3 — Load LDA per-video topic probabilities, prefix by community

In [ ]:
lda_parts = []
for community, short_name in COMMUNITY_SHORT_NAME.items():
    pattern = os.path.join(LDA_FINAL_DF_DIR, f"df_LDA_{short_name}.csv")
    matches = glob.glob(pattern)

    if not matches:
        print(f"WARNING: no per-video LDA file found for {community} (pattern: {pattern})")
        continue

    part_df = pd.read_csv(matches[0])

    # prefix every Topic_N_Probability column with the community name
    rename_map = {
        col: f"{short_name}_{col}" for col in part_df.columns
        if col.startswith("Topic_") and col.endswith("_Probability")
    }
    part_df = part_df.rename(columns=rename_map)

    lda_parts.append(part_df)
    print(f"{community}: {len(rename_map)} topic columns loaded, {len(part_df)} videos")

# outer concat: each video only has values in its own community's topic columns,
# NaN everywhere else (correctly means "not applicable", not "zero probability")
lda_wide_df = pd.concat(lda_parts, ignore_index=True, sort=False)
lda_wide_df = lda_wide_df.drop(columns=["community"], errors="ignore")

print(f"\nCombined LDA wide df: {lda_wide_df.shape}")
print(f"Duplicate video_ids: {lda_wide_df['video_id'].duplicated().sum()}")

transcripts_business: 5 topic columns loaded, 1043 videos
transcripts_religion: 15 topic columns loaded, 1033 videos
transcripts_comedy: 5 topic columns loaded, 1063 videos
transcripts_lifestyle: 5 topic columns loaded, 1113 videos
transcripts_tech: 30 topic columns loaded, 1434 videos
politics_transcripts: 10 topic columns loaded, 1357 videos
gaming_transcripts: 5 topic columns loaded, 1157 videos
motivational_transcripts: 5 topic columns loaded, 1006 videos

Combined LDA wide df: (9206, 90)
Duplicate video_ids: 0


Cell 4 — Merge everything

In [ ]:
before_shape = df.shape

df_merged = df.merge(stanza_features_df, on="video_id", how="left", suffixes=("", "_stanza_dup"))
df_merged = df_merged.merge(gender_df, on="video_id", how="left", suffixes=("", "_gender_dup"))
df_merged = df_merged.merge(lda_wide_df, on="video_id", how="left", suffixes=("", "_lda_dup"))

assert len(df_merged) == before_shape[0], "Row count changed — a feature source has duplicate video_ids"

dup_cols = [c for c in df_merged.columns if c.endswith(("_stanza_dup", "_gender_dup", "_lda_dup"))]
if dup_cols:
    print(f"WARNING: unexpected overlapping columns: {dup_cols}")

print(f"Before merge: {before_shape}")
print(f"After merge: {df_merged.shape}")

Before merge: (9206, 11)
After merge: (9206, 177)


Cell 5 — Null audit, column by column

In [ ]:
print("--- Null counts per column (only columns with any nulls shown) ---\n")
null_summary = df_merged.isna().sum()
null_summary = null_summary[null_summary > 0].sort_values(ascending=False)

null_pct = (null_summary / len(df_merged) * 100).round(1)
null_report = pd.DataFrame({"null_count": null_summary, "null_pct": null_pct})
print(null_report.to_string())

print(f"\n--- Rows missing gender data (male_seconds/female_seconds/dominant_gender) ---")
gender_cols = [c for c in ["male_seconds", "female_seconds", "dominant_gender"] if c in df_merged.columns]
if gender_cols:
    missing_gender_mask = df_merged[gender_cols[0]].isna()
    print(f"{missing_gender_mask.sum()} rows missing gender data")
    print(df_merged.loc[missing_gender_mask, "community"].value_counts())

--- Null counts per column (only columns with any nulls shown) ---

                                  null_count  null_pct
motivational_Topic_4_Probability        8200      89.1
motivational_Topic_3_Probability        8200      89.1
motivational_Topic_1_Probability        8200      89.1
motivational_Topic_2_Probability        8200      89.1
motivational_Topic_5_Probability        8200      89.1
religion_Topic_11_Probability           8173      88.8
religion_Topic_9_Probability            8173      88.8
religion_Topic_10_Probability           8173      88.8
religion_Topic_6_Probability            8173      88.8
religion_Topic_5_Probability            8173      88.8
religion_Topic_4_Probability            8173      88.8
religion_Topic_13_Probability           8173      88.8
religion_Topic_14_Probability           8173      88.8
religion_Topic_12_Probability           8173      88.8
religion_Topic_8_Probability            8173      88.8
religion_Topic_7_Probability            8173      88

Cell 6 — Remove rows with no gender segment, row-count audit


In [ ]:
before_drop = len(df_merged)
before_by_community = df_merged["community"].value_counts()

gender_cols = [c for c in ["male_seconds", "female_seconds", "dominant_gender"] if c in df_merged.columns]
bad_mask = df_merged[gender_cols[0]].isna() | (df_merged["dominant_gender"] == "unknown")
df_clean = df_merged[~bad_mask].copy()

after_drop = len(df_clean)
after_by_community = df_clean["community"].value_counts()

print(f"Rows before gender-null drop: {before_drop}")
print(f"Rows after gender-null drop: {after_drop}")
print(f"Rows removed: {before_drop - after_drop}\n")

print("--- Row counts per community (before vs after cleaning) ---")
comparison = pd.DataFrame({"before": before_by_community, "after": after_by_community})
comparison["dropped"] = comparison["before"] - comparison["after"]
comparison["dropped_pct"] = (comparison["dropped"] / comparison["before"] * 100).round(1)
print(comparison.to_string())

Rows before gender-null drop: 9206
Rows after gender-null drop: 8847
Rows removed: 359

--- Row counts per community (before vs after cleaning) ---
                          before  after  dropped  dropped_pct
community                                                    
transcripts_business        1043    974       69          6.6
transcripts_comedy          1063   1059        4          0.4
transcripts_gaming          1157   1042      115          9.9
transcripts_lifestyle       1113   1112        1          0.1
transcripts_motivational    1006    991       15          1.5
transcripts_politics        1357   1357        0          0.0
transcripts_religion        1033    882      151         14.6
transcripts_tech            1434   1430        4          0.3


# Cell 7 — Save final clean df


In [ ]:
columns_to_drop = [col for col in df_clean.columns if "Unnamed" in col]
df_clean = df_clean.drop(columns=columns_to_drop)
df_clean["transcript"] = df_clean["transcript"].fillna("")

output_path = os.path.join(OUTPUT_DIR, "df-all-features.csv")
df_clean.to_csv(output_path, index=False)

print(f"Saved final clean all-features df: {df_clean.shape}")
print(f"\nFinal row counts per community:")
print(df_clean["community"].value_counts())

Saved final clean all-features df: (8847, 178)

Final row counts per community:
community
transcripts_tech            1430
transcripts_politics        1357
transcripts_lifestyle       1112
transcripts_comedy          1059
transcripts_gaming          1042
transcripts_motivational     991
transcripts_business         974
transcripts_religion         882
Name: count, dtype: int64


================= Final verification of df-all-features.csv =================


In [ ]:
output_path

'/content/drive/MyDrive/Transcripts_CSS/outputs/df-all-features.csv'

In [ ]:
check_df = pd.read_csv(output_path)

print(f"Loaded saved file: {check_df.shape}")
print(f"Expected: no NaN/unknown in gender columns, no duplicate video_ids\n")

# ---- 1. Shape & row count sanity ----
print(f"--- Shape check ---")
print(f"Rows: {check_df.shape[0]}, Columns: {check_df.shape[1]}")
print(f"(before drop was {before_drop}, expected drop = {before_drop - after_drop}, "
      f"so this should equal {after_drop})\n")

# ---- 2. Duplicate video_id check ----
print(f"--- Duplicate video_id check ---")
n_dup = check_df["video_id"].duplicated().sum()
print(f"Duplicate video_ids: {n_dup}")
if n_dup > 0:
    print(check_df[check_df["video_id"].duplicated(keep=False)].sort_values("video_id")[["video_id", "community"]].to_string(index=False))
print()

# ---- 3. Gender column integrity: no NaN, no 'unknown' should remain ----
print(f"--- Gender column integrity ---")
print("dominant_gender value counts:")
print(check_df["dominant_gender"].value_counts(dropna=False).to_string())
n_gender_null = check_df["dominant_gender"].isna().sum()
n_gender_unknown = (check_df["dominant_gender"] == "unknown").sum()
print(f"\nNull: {n_gender_null} (expected 0)")
print(f"Unknown: {n_gender_unknown} (expected 0)")
print()

# ---- 4. Community-wise row counts ----
print(f"--- Community-wise row counts (final) ---")
print(check_df["community"].value_counts().to_string())
print()

# ---- 5. Community-wise gender breakdown (male/female balance check) ----
print(f"--- Community-wise gender breakdown ---")
print(pd.crosstab(check_df["community"], check_df["dominant_gender"]).to_string())
print()

# ---- 6. Full null audit again, but split into "expected" (topic cols) vs "unexpected" ----
print(f"--- Null audit on final saved df ---")
null_summary = check_df.isna().sum()
null_summary = null_summary[null_summary > 0].sort_values(ascending=False)
null_pct = (null_summary / len(check_df) * 100).round(1)
null_report = pd.DataFrame({"null_count": null_summary, "null_pct": null_pct})

topic_null_cols = [c for c in null_report.index if "_Topic_" in c and c.endswith("_Probability")]
non_topic_null_cols = [c for c in null_report.index if c not in topic_null_cols]

print(f"Topic-probability columns with nulls (expected — structural, not a problem): {len(topic_null_cols)} columns")
print(f"\nNON-topic columns with nulls (worth checking each of these):")
if non_topic_null_cols:
    print(null_report.loc[non_topic_null_cols].to_string())
else:
    print("None — clean!")
print()

# ---- 7. 'Unnamed' leftover columns check ----
print(f"--- 'Unnamed' column check ---")
unnamed_cols = [c for c in check_df.columns if "Unnamed" in c]
print(f"Leftover 'Unnamed' columns: {unnamed_cols if unnamed_cols else 'None — clean!'}")
print()

# ---- 8. transcript column check (should have no NaN after fillna("")) ----
print(f"--- transcript column check ---")
if "transcript" in check_df.columns:
    n_transcript_null = check_df["transcript"].isna().sum()
    n_transcript_empty = (check_df["transcript"] == "").sum()
    print(f"Null transcripts: {n_transcript_null} (expected 0)")
    print(f"Empty-string transcripts: {n_transcript_empty}")
print()

# ---- 9. Cross-check against comparison table from earlier ----
print(f"--- Community drop comparison (recap from Cell 8) ---")
print(comparison.to_string())

Loaded saved file: (8847, 178)
Expected: no NaN/unknown in gender columns, no duplicate video_ids

--- Shape check ---
Rows: 8847, Columns: 178
(before drop was 9206, expected drop = 359, so this should equal 8847)

--- Duplicate video_id check ---
Duplicate video_ids: 0

--- Gender column integrity ---
dominant_gender value counts:
dominant_gender
male      5232
female    3615

Null: 0 (expected 0)
Unknown: 0 (expected 0)

--- Community-wise row counts (final) ---
community
transcripts_tech            1430
transcripts_politics        1357
transcripts_lifestyle       1112
transcripts_comedy          1059
transcripts_gaming          1042
transcripts_motivational     991
transcripts_business         974
transcripts_religion         882

--- Community-wise gender breakdown ---
dominant_gender           female  male
community                             
transcripts_business         332   642
transcripts_comedy           346   713
transcripts_gaming           417   625
transcripts_lifestyl